# AND(empiar10311_stack_crop)

In [ ]:
from pathlib import Path
from collections import namedtuple

In [ ]:
Args = namedtuple("args", ["input", "output1", "output2"])
args = Args("empiar10311_stack_crop.mrc",
            "empiar10311_stack_crop__AND.mrc",
            "empiar10311_stack_crop__AND.pdf")

In [ ]:
from my_google_auth import DriveHandler
service = DriveHandler.get_drive_service()
handler = DriveHandler.DriveHandler(service)

In [ ]:
DRIVE_TOMOGRADENOISING_TMP       = '1hGHvkP46fxLCQbUlyYhAS_eVl6PollQM'  # "tmp" folder
DRIVE_TOMOGRADENOISING_TOMOGRAMS = '1hfAOv6etLjB16K-nCrg-0u24iZBvmr1-'  # "Tomograms" folder

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        file_id = handler.find_file_id(drive_file_name=output, drive_folder_id=DRIVE_TOMOGRADENOISING_TMP)
        if file_id == None:
            print(f"{output} does not exist in Google Drive. Creating ...")
        else:
            print(f"Downloading {output} from Google Drive")
            success = handler.download(file_id, local_save_path=output)

In [ ]:
import mrcfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from medpy.filter.smoothing import anisotropic_diffusion
from my_google_auth import DriveHandler
from collections import namedtuple

In [ ]:
file_path = Path(args.input)
if file_path.exists():
    print(f"Found local {args.input}")
else:
    file_id = handler.find_file_id(drive_file_name=args.input, drive_folder_id=DRIVE_TOMOGRADENOISING_TOMOGRAMS)
    success = handler.download(file_id, local_save_path=args.input)

In [ ]:
with mrcfile.open(args.input, permissive=True) as mrc:
    vol = mrc.data.astype(np.float32)

In [ ]:
vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-8)

denoised = anisotropic_diffusion(
    vol,
    niter=20,     # number of iterations (10–30) (More iterations = stronger smoothing)
    kappa=30,     # conduction coefficient (Edge detection sensitivity) (20–40) (Larger → smoother but weaker edges)
    gamma=0.1,    # time step (Step size) (stability) (0.05–0.15) (Must be ≤0.25)
    option=1,     # PM1 model (Conduction model) (1 or 2) (1 = sharper edges preserved)
    voxelspacing=None,  # use if voxel spacing differs per axis (Voxel geometry) (None or (sx,sy,sz)) (Use if voxels are anisotropic)
)

In [ ]:
denoised = anisotropic_diffusion(
    vol,          # [0.0, 1.0]!
    niter=12,     # number of iterations (10–30) (More iterations = stronger smoothing)
    kappa=15,     # conduction coefficient (Edge detection sensitivity) (20–40) (Larger → smoother but weaker edges)
    gamma=0.07,   # time step (Step size) (stability) (0.05–0.15) (Must be ≤0.25)
    option=1,     # PM1 model (Conduction model) (1 or 2) (1 = sharper edges preserved)
    voxelspacing=None,  # use if voxel spacing differs per axis (Voxel geometry) (None or (sx,sy,sz)) (Use if voxels are anisotropic)
)

In [ ]:
denoised.shape

In [ ]:
Z_dim = vol.shape[0]
Z2 = Z_dim//2

In [ ]:
with mrcfile.new(args.output1, overwrite=True) as mrc:
    mrc.set_data(denoised.astype(np.float32))
    mrc.data

In [ ]:
def read_MRC(file_path):
    return mrcfile.read(file_path)

In [ ]:
denoised = read_MRC(args.output1)

In [ ]:
figure(figsize=(32, 32))
plt.subplot(1, 3, 1)
plt.title("original")
imgplot = plt.imshow(vol[7][::-1, :], cmap="gray")
plt.subplot(1, 3, 2)
plt.title("FlowDenoising")
plt.imshow(denoised[7][::-1, :], cmap="gray")
plt.subplot(1, 3, 3)
plt.title("difference")
plt.imshow(vol[7][::-1, :] - denoised[7][::-1, :], cmap="gray")

In [ ]:
from matplotlib.pyplot import figure
figure(figsize=(16, 16))
slice_idx = denoised.shape[0]//2
plt.imshow(denoised[slice_idx, 200:600, 200:600], cmap="gray")
plt.savefig(args.output2, bbox_inches='tight')

In [ ]:
plt.close()

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        uploaded_file_id = handler.upload(
            local_file_path=output,
            drive_file_name=output,
            drive_folder_id=DRIVE_TOMOGRADENOISING_TMP)